### Notebook to create BLOCK-T377 for Camera Hexapod Sensitivity

Created on: 2025-03-24

Author: Guillem Megias

In [ ]:
from lsst.ts.observing import ObservingBlock, ObservingScript 
from lsst.ts.aos.analysis import build_configuration_schema
import os

In [ ]:
current_path = os.getcwd()
block_number = 'T377'
program = "BLOCK-T377"
reason = "SITCOM-820"
note = "sensitivity_cam_"
constraints = []

### Define configuration schema

In [ ]:
# Define the configurable properties that we will use in the configuration schema
properties = {
    "filter": {
        "description": "Filter to use.",
        "type": "string",
        "default": "r_57"
    },
    "num_steps": {
        "description": "Number of steps for the sensitivity matrix.",
        "type": "integer",
        "default": 9
    },
    "exp_time": {
        "description": "Exposure time.",
        "type": "float",
        "default": 15.0
    },
    "rotation_sequence": {
        "description": "Rotation sequence.",
        "type": "array",
        "items": {
            "type": "float"
        },
        "default": [-75.0, -63.75, -42.5, -21.25, 0.0, 21.25, 42.5, 63.75, 75.0]
    },
    "el": {
        "description": "Elevation.",
        "type": "float",
        "default": 60.0
    },
    "az": {
        "description": "Azimuth.",
        "type": "float",
        "default": 0.0
    },
}

# Build the configuration schema for BLOCK-404
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

### Define scripts and block

In [ ]:
ranges = [150, 5000, 5000, 0.1, 0.1]
labels = ['dz', 'dx', 'dy', 'drx', 'dry']
scripts = []

for idx, range in enumerate(ranges):
    parameter_march_script = ObservingScript(
        name="maintel/parameter_march_lsstcam.py",
        standard=True,
        parameters= dict(
            exp_time="$exp_time",
            dof_index=idx + 5,
            rotation_sequence="$rotation_sequence",
            range=range,
            n_steps="$num_steps",
            program="$program",
            filter="$filter",
            reason=reason,
            note=f"sensitivity_cam_{labels[idx]}",
            az="$az",
            el="$el",
        )
    )
    scripts.append(parameter_march_script)

In [ ]:
block = ObservingBlock(
    name = program,
    program = program,
    configuration_schema=configuration_schema,
    scripts = scripts,
)

### Save configurable block

In [ ]:
block.model_dump_json(indent=2)

output_file_path = f'{current_path}/aos/ts_config_ocs/Scheduler/observing_blocks_maintel/AOS/sensitivity_matrix/{program}.json'

with open(output_file_path, 'w') as file:
    file.write(block.model_dump_json(indent=2))